In [2]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
conn=psycopg2.connect(
    database='olist_db',
    host='localhost',
    port='5433',
    user='postgres',
    password='1234'
    )

engine = create_engine('postgresql+psycopg2://', creator=lambda: conn)



In [3]:
# analysing null values 

tables=['customer','geolocation','order_items','orders','payments','product_category','products','reviews','sellers']


for table in tables:
    df=pd.read_sql(f"SELECT * FROM {table};",engine)
    print(f" Table   :{table}")
    print(f" Rows   :{len(df)}")
    print(f" Nulls  :{df.isnull().sum().sum()}")
    print(f" Duplicates  :{df.duplicated().sum()}")


# Table   :geolocation
#  Rows   :1000163
#  Nulls  :0
#  Duplicates  :261831

#  Table   :orders
#  Rows   :99441
#  Nulls  :4908
#  Duplicates  :0

# Table   :products
#  Rows   :32951
#  Nulls  :2448
#  Duplicates  :0

# Table   :reviews
#  Rows   :99224
#  Nulls  :145903
#  Duplicates  :0



 Table   :customer
 Rows   :99442
 Nulls  :99441
 Duplicates  :0
 Table   :geolocation
 Rows   :1000163
 Nulls  :0
 Duplicates  :261831
 Table   :order_items
 Rows   :112650
 Nulls  :0
 Duplicates  :0
 Table   :orders
 Rows   :99441
 Nulls  :4908
 Duplicates  :0
 Table   :payments
 Rows   :103886
 Nulls  :0
 Duplicates  :0
 Table   :product_category
 Rows   :71
 Nulls  :0
 Duplicates  :0
 Table   :products
 Rows   :32951
 Nulls  :2448
 Duplicates  :0
 Table   :reviews
 Rows   :99224
 Nulls  :145903
 Duplicates  :0
 Table   :sellers
 Rows   :3095
 Nulls  :0
 Duplicates  :0


In [4]:
#  Table   :orders
#  Rows   :99441
#  Nulls  :4908
#  Duplicates  :0

df=pd.read_sql("SELECT * FROM sellers;",engine)
df.isnull().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [5]:
df.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [6]:
df['order_purchase_timestamp']=pd.to_datetime(df['order_purchase_timestamp'])
df['order_approved_at']=pd.to_datetime(df['order_approved_at'])
df['order_delivered_carrier_date']=pd.to_datetime(df['order_delivered_carrier_date'])
df['order_delivered_customer_date']=pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date']=pd.to_datetime(df['order_estimated_delivery_date'])



KeyError: 'order_purchase_timestamp'

In [ ]:
df['order_approved_at'] = df['order_approved_at'].ffill()
df['order_delivered_carrier_date'] = df['order_delivered_carrier_date'].ffill()
df['order_delivered_customer_date'] = df['order_delivered_customer_date'].ffill()

df.isnull().sum()

In [ ]:

# Table   :products
#  Rows   :32951
#  Nulls  :2448
#  Duplicates  :0


df=pd.read_sql("SELECT * FROM products;",engine)

df.isnull().sum()


df['product_category_name']=df['product_category_name'].fillna(df['product_category_name'].mode()[0])

df['product_name_lenght']=df['product_name_lenght'].fillna(df['product_name_lenght'].mean())

df['product_description_lenght']=df['product_description_lenght'].fillna(df['product_description_lenght'].mean())

df['product_photos_qty']=df['product_photos_qty'].fillna(df['product_photos_qty'].mean())

df['product_weight_g']=df['product_weight_g'].fillna(df['product_weight_g'].mean())


df['product_length_cm']=df['product_length_cm'].fillna(df['product_length_cm'].mean())


df['product_width_cm']=df['product_width_cm'].fillna(df['product_width_cm'].mean())

df['product_height_cm']=df['product_height_cm'].fillna(df['product_height_cm'].mean())

df



In [ ]:

# Table   :reviews
#  Rows   :99224
#  Nulls  :145903
#  Duplicates  :0


df=pd.read_sql("SELECT * FROM orders;",engine)
df

In [ ]:
df=df.drop(columns=['review_comment_title','review_comment_message'])

df

In [ ]:
# joining the tables 

df=pd.read_sql("""SELECT *  FROM  orders""",engine)
df


In [7]:
df=pd.read_sql("""
        SELECT o.order_purchase_timestamp,c.customer_id,r.review_score,o.order_status,o.order_estimated_delivery_date,o.order_delivered_customer_date,c.customer_state,c.customer_name,oi.price,p.payment_value,p.payment_type,pr.product_category_name
        FROM orders o
        LEFT JOIN customer c ON c.customer_id=o.customer_id
        LEFT JOIN order_items oi ON o.order_id=oi.order_id
        LEFT JOIN payments p ON p.order_id=o.order_id
        LEFT JOIN products pr ON pr.product_id=oi.product_id
        LEFT JOIN reviews r ON r.order_id=o.order_id
""",engine)


df['order_purchase_date']=df['order_purchase_timestamp'].str.split(' ').str[0]

df['order_purchase_date']=pd.to_datetime(df['order_purchase_date'])

df

,order_purchase_timestamp,customer_id,review_score,order_status,order_estimated_delivery_date,order_delivered_customer_date,customer_state,customer_name,price,payment_value,payment_type,product_category_name,order_purchase_date
0,2017-12-15 00:06:10,f5afca14dfa9dc64251cf2b45c54c363,4.0,delivered,2018-01-16 00:00:00,2018-01-03 15:09:32,RJ,None,369.00,386.33,credit_card,relogios_presentes,2017-12-15
1,2017-10-02 10:56:33,9ef432eb6251297304e76186b10a928d,4.0,delivered,2017-10-18 00:00:00,2017-10-10 21:25:13,SP,None,29.99,18.59,voucher,utilidades_domesticas,2017-10-02
2,2017-10-02 10:56:33,9ef432eb6251297304e76186b10a928d,4.0,delivered,2017-10-18 00:00:00,2017-10-10 21:25:13,SP,None,29.99,2.00,voucher,utilidades_domesticas,2017-10-02
3,2017-10-02 10:56:33,9ef432eb6251297304e76186b10a928d,4.0,delivered,2017-10-18 00:00:00,2017-10-10 21:25:13,SP,None,29.99,18.12,credit_card,utilidades_domesticas,2017-10-02
4,2017-01-23 18:29:09,f54a9f0e6b351c431402b8461ea51999,1.0,delivered,2017-03-06 00:00:00,2017-02-02 14:08:10,RS,None,19.90,35.95,boleto,moveis_decoracao,2017-01-23
...,...,...,...,...,...,...,...,...,...,...,...,...,...
119138,2018-08-10 12:24:44,7794a175e2771699e4dd7b1e141451e0,1.0,delivered,2018-08-27 00:00:00,2018-08-17 14:28:48,MG,None,35.90,55.34,boleto,bebes,2018-08-10
119139,2018-03-10 22:13:00,8eda31e4efd069e42b3a3bfd80ac5b7b,4.0,delivered,2018-03-28 00:00:00,2018-03-26 22:22:10,SP,None,274.90,289.26,credit_card,relogios_presentes,2018-03-10
119140,2018-01-22 10:55:26,640cc2572fb312376699744ed394f935,3.0,delivered,2018-02-15 00:00:00,2018-02-01 22:25:26,SP,None,149.00,161.54,credit_card,informatica_acessorios,2018-01-22
119141,2018-08-21 12:21:00,c4c369211d1aaab90c8e097f6939dda2,3.0,unavailable,2018-08-29 00:00:00,None,SP,None,NaN,268.74,credit_card,None,2018-08-21


In [8]:
# rfm segment calculation 
refrence_date=df['order_purchase_date'].max()

rfm=df.groupby('customer_id').agg(
           recency=('order_purchase_date',(lambda x:(refrence_date-x.max()).days)),
            frequancy=('order_purchase_date','count'),
            monetary=('price','sum')
).reset_index()

rfm['status']=rfm['recency'].apply(lambda x: 'Churn' if x>=90 else 'active') 

rfm.columns=['customer_id','recency','frequancy','monetary','status']

rfm[rfm['status']=='active']

,customer_id,recency,frequancy,monetary,status
9,000598caf2ef4117407665ac33275130,67,1,1107.00,active
41,001df1ee5c36767aa607001ab1a13a06,73,1,29.99,active
50,002554bdf9eb99618d8189c3a89fdd52,65,1,229.00,active
51,0026955706fd4e2fa997f3f4c18d485a,58,1,99.90,active
55,002905287304e28c0218389269b4759b,87,1,13.47,active
...,...,...,...,...,...
99346,ffbd58aa41cbab07f08190801e939f59,80,6,53.28,active
99352,ffc0c21bf66cb129c62d8f89ed19c26c,87,1,1389.99,active
99398,ffe1eab23bff108bf37c973b05d4e9ba,60,1,79.99,active
99414,fff168ca1f8a1d2e8e2108b231a68a8c,89,3,47.70,active


90001 means 90% are churned customer and 9440 10% are active customers

In [9]:
# Customer Segmentation RFM Quintile Scoring


rfm['R_score']=pd.qcut(rfm['recency'],q=5,labels=[5,4,3,2,1])

rfm['F_score']=pd.qcut(rfm['frequancy'].rank(method='first'),q=5,labels=[1,2,3,4,5])

rfm['M_score']=pd.qcut(rfm['monetary'].rank(method='first'),q=5,labels=[1,2,3,4,5])

rfm['RFM_score']=rfm['R_score'].astype(str)+rfm['F_score'].astype(str)+rfm['M_score'].astype(str)



def assignment(row):


    r=int(row['R_score'])
    f=int(row['F_score'])
    m=int(row['M_score'])


    if r >= 4 and f >= 4 and m >= 4:
        return 'Vip customers'          # best customers
    elif r >= 4 and f >= 3:
        return 'Loyal'              # regular customers
    elif r >= 3 and f <= 2:
        return 'Potential Loyalist' # naye customers
    elif r <= 2 and f >= 3:
        return 'At Risk'            # pehle achhe the ab nahi aate
    elif r <= 2 and f <= 2:
        return 'Lost'               # gone case
    else:
        return 'Others'

rfm['Segment']=rfm.apply(assignment,axis=1)

print(rfm['Segment'].value_counts())

print(rfm[['customer_id','recency','frequancy','monetary','status','R_score','F_score','M_score','RFM_score','Segment']].head(22))




Segment
Potential Loyalist    24078
At Risk               24058
Loyal                 16605
Lost                  15699
Others                11765
Vip customers          7236
Name: count, dtype: int64
                         customer_id  recency  frequancy  monetary  status  \
0   00012a2ce6f8dcda20d059ce98491703      337          1     89.80   Churn   
1   000161a058600d5901f007fab4c27140      458          1     54.90   Churn   
2   0001fd6190edaaf884bcaf3d49edf079      596          1    179.99   Churn   
3   0002414f95344307404f0ace7a26f1d5      427          1    149.90   Churn   
4   000379cdec625522490c315e70c7a9fb      198          1     93.00   Churn   
5   0004164d20a9e969af783496f3408652      553          1     59.99   Churn   
6   000419c5494106c306a97b5635748086      229          1     34.30   Churn   
7   00046a560d407e99b969756e0b10f282      303          1    120.90   Churn   
8   00050bf6e01e69d5c0fd612f1bcfb69c      395          1     69.99   Churn   
9   000598caf2ef41

/var/folders/7l/jc129jxx3n5dwdmjkbcxb_f40000gn/T/ipykernel_38377/1201889328.py:41: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df[rfm['F_score']>5]


IndexingError: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).

# 📊 RFM Segmentation — Business Insights

---

## 📦 Segment Overview

| Segment | Customers | Percentage | Priority |
|---|---|---|---|
| 🌱 Potential Loyalist | 24,052 | 24% | 🟡 HIGH |
| ⚠️ At Risk | 24,032 | 24% | 🔴 HIGH |
| 💛 Loyal | 16,632 | 17% | 🟢 MEDIUM |
| 💀 Lost | 15,725 | 16% | ⚫ LOW |
| 🔘 Others | 11,748 | 12% | 🔵 MEDIUM |
| 👑 VIP | 7,252 | 7% | 🟣 HIGH |
| **Total** | **99,441** | **100%** | |

---

## 🔴 Biggest Problem — At Risk (24%)

> ⚡ **Priority No. 1** — Saving these customers is most urgent

**Who are they?**
- 24,032 customers who used to shop regularly
- They liked your store — but **stopped coming**
- Good frequency in the past — but recency is very low

**Action Plan:**

| Step | Action | Timeline |
|---|---|---|
| 1 | Send "We miss you" email with **20-30% discount** | Immediately |
| 2 | If no response → Send **final offer** | After 2 weeks |
| 3 | If still no response → Move to **Lost** | After 4 weeks |

---

## 🌱 Biggest Opportunity — Potential Loyalist (24%)

> ⚡ **Priority No. 2** — These can become your loyal customers

**Who are they?**
- 24,052 new customers who came **recently**
- They liked the experience — but only visited **1-2 times**
- High potential to become loyal

**Action Plan:**

| Step | Action | Timeline |
|---|---|---|
| 1 | Send **welcome email** immediately | Day 1 |
| 2 | Give **10-15% discount** on second order | Day 3 |
| 3 | Keep engaging — push them to buy again | Week 2 |

---

## 💛 Stable Base — Loyal (17%)

> ⚡ These are your **backbone** — never lose them

**Who are they?**
- 16,632 customers who come back regularly
- Good recency **and** good frequency
- They trust your brand

**Action Plan:**

| Step | Action |
|---|---|
| 1 | Start a **loyalty points program** |
| 2 | Give **early access** to new products |
| 3 | Push them towards becoming **VIP** |

---

## 👑 Most Valuable — VIP (7%)

> ⚡ Only 7% customers — but probably driving **40-50% of your revenue!**

**Who are they?**
- Only 7,252 customers
- Came recently + come frequently + spend the most money
- Your **best customers**

**Action Plan:**

| Step | Action |
|---|---|
| 1 | Send **personal thank you** messages |
| 2 | Give **exclusive VIP-only** offers |
| 3 | Make them feel **special always** |

---

## 💀 Gone Case — Lost (16%)

> ⚡ Do not waste too much budget here

**Who are they?**
- 15,725 customers who came long ago — **never returned**
- Low frequency too — experience was not great
- Very hard to bring back

**Action Plan:**

| Step | Action |
|---|---|
| 1 | One **last attempt** — give big 40-50% discount |
| 2 | If no response → **stop marketing** to them |
| 3 | Focus budget on better segments |

---

## 🔘 Need Analysis — Others (12%)

> ⚡ Need a deeper look

**Who are they?**
- 11,748 customers who did not fit any segment
- They are in the **middle ground**

**Action Plan:**

| Step | Action |
|---|---|
| 1 | Manually check their **R, F, M values** |
| 2 | Re-assign to **correct segment** |
| 3 | Build a **separate strategy** for them |

---

## ⚡ Top 3 Actions — Do This Now

```
1️⃣  At Risk         → Launch win-back campaign    (save 24% customers)
2️⃣  Potential       → Send onboarding emails      (convert 24% to loyal)
3️⃣  VIP             → Give special treatment      (protect your revenue)
```

---

## 💡 Key Takeaway

> **If you can only do one thing — Target At Risk customers first**
>
> Because they were good customers before — bringing them back is **easiest and most profitable!**

In [ ]:
# df['delievery_delay']=(df['order_estimated_delivery_date']-df['order_delivered_customer_date']).dt.days



df['order_estimated_delivery_date']=pd.to_datetime(df['order_estimated_delivery_date'])
df['order_delivered_customer_date']=pd.to_datetime(df['order_delivered_customer_date'])


df['delivery_delay']=(df['order_estimated_delivery_date'] - df['order_delivered_customer_date']).dt.days

customer_delivery_delay=df.groupby('customer_id')['delivery_delay'].mean().reset_index()

late_delivery_customers=customer_delivery_delay[customer_delivery_delay['delivery_delay'] > 0 ]
print(f"Total Late Delivery Customers: {len(late_delivery_customers)}")


rfm_delay=rfm.merge(customer_delivery_delay,how='left', on='customer_id')

churn_customers=rfm_delay[rfm_delay['Segment'].isin(['At Risk','Lost'])]

print(f" total churn customers  {len(churn_customers)}")

late_churn=churn_customers[churn_customers['delivery_delay']>0]
print(f"late churn customers  {len(late_churn)}")

average_delay_vip=rfm_delay[rfm_delay['Segment']=='Vip customers']['delivery_delay'].mean()

print(average_delay_vip)

In [ ]:
# pareto principal for customers

top_customers = df.groupby('customer_id')['payment_value'].sum().reset_index()
top_customers = top_customers.sort_values('payment_value', ascending=False)

n = round(len(top_customers) * 0.20)
top_20 = top_customers.head(n)
print(top_20)

In [ ]:
import matplotlib.pyplot as plt

# Sirf top 20 rows lo
top_20_plot = top_customers.head(20)

plt.figure(figsize=(12, 6))
plt.barh(range(20), top_20_plot['payment_value'])  # barh = horizontal bars, easy to read
plt.yticks(range(20), [f"Customer {i+1}" for i in range(20)])
plt.xlabel("Payment Value")
plt.ylabel("Customer Rank")
plt.title("Top 20 Customers by Payment Value")
plt.tight_layout()
plt.show()

In [ ]:
churn_customers=rfm[rfm['status']=='Churn']
active_customers=rfm[rfm['status']=='active']


print(f"total churn customers {len(churn_customers)}")
print(f"total active customers {len(active_customers)}")



churn_id=churn_customers['customer_id']

churn_customer_amount=df[df['customer_id'].isin(churn_id)].groupby('customer_id')['payment_value'].sum().reset_index()

print(f"Total churn amount: {churn_customer_amount['payment_value'].sum():.2f}")

active_id=active_customers['customer_id']
active_customer_amount=df[df['customer_id'].isin(active_id)].groupby('customer_id')['payment_value'].sum().reset_index()

total_payment=df['payment_value'].sum()

print(f"Total active amount: {active_customer_amount['payment_value'].sum():.2f}")

print(f" total payment {total_payment}")




In [ ]:
# converting data frame in sql


df.to_sql('main_data', engine, if_exists='replace', index=False)
rfm.to_sql('rfm_data', engine, if_exists='replace', index=False)

print("Data PostgreSQL mein save ho gaya!")